In [ ]:
!pip -q install fastf1 pandas numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.3 MB/s eta 0:00:00


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import fastf1
from urllib.request import urlretrieve

In [ ]:
import json
import numpy as np
import pandas as pd


class FrenetConverter:
    """
    用途:
        1) 用赛道 centerline 定义 Frenet 坐标
        2) 用 raceline / centerline 和 FastF1 最快圈做几何对齐
        3) 直接处理某个车手完整比赛 telemetry 流
        4) 导出按 SessionTime 排序的整场比赛 CSV

    track_df 需要列:
        x_m, y_m, w_tr_right_m, w_tr_left_m

    raceline_df 需要列:
        x_m, y_m
    """

    def __init__(self, track_df, raceline_df=None, xy_cols=("X", "Y"), n_samples=800):
        self.track_df = track_df.copy()
        self.track_xy = self.track_df[["x_m", "y_m"]].to_numpy(dtype=float)

        if raceline_df is None:
            self.raceline_df = None
            self.align_xy = self.track_xy
        else:
            self.raceline_df = raceline_df.copy()
            self.align_xy = self.raceline_df[["x_m", "y_m"]].to_numpy(dtype=float)

        self.xy_cols = xy_cols
        self.n_samples = n_samples

        self.track_s_ref, self.track_len = self._cumulative_arc_length(self.track_xy, closed=True)

        self.track_tangents = self._compute_centerline_tangents(self.track_xy, closed=True)
        self.track_normals = self._compute_centerline_normals(self.track_xy, closed=True)
        self.track_curvature = self._compute_centerline_curvature(self.track_xy, closed=True)

        self._build_segment_cache()

        self.is_fitted = False
        self.scale = None
        self.R = None
        self.src_center = None
        self.dst_center = None
        self.fit_error = None

    @staticmethod
    def _cumulative_arc_length(points, closed=True):
        pts = np.asarray(points, dtype=float)
        diffs = np.diff(pts, axis=0)
        seg_lens = np.linalg.norm(diffs, axis=1)

        s_ref = np.zeros(len(pts))
        s_ref[1:] = np.cumsum(seg_lens)

        total_len = s_ref[-1]
        if closed and len(pts) > 1:
            total_len += np.linalg.norm(pts[0] - pts[-1])

        return s_ref, total_len

    @staticmethod
    def _resample_closed_curve(points, n_samples=800):
        pts = np.asarray(points, dtype=float)
        pts_closed = np.vstack([pts, pts[0]])

        diffs = np.diff(pts_closed, axis=0)
        seg_lens = np.linalg.norm(diffs, axis=1)

        s = np.zeros(len(pts_closed))
        s[1:] = np.cumsum(seg_lens)
        total_len = s[-1]

        s_new = np.linspace(0, total_len, n_samples, endpoint=False)
        x_new = np.interp(s_new, s, pts_closed[:, 0])
        y_new = np.interp(s_new, s, pts_closed[:, 1])

        return np.column_stack([x_new, y_new]), total_len

    @staticmethod
    def _best_rotation_kabsch(A, B):
        H = A.T @ B
        U, _, Vt = np.linalg.svd(H)
        R = Vt.T @ U.T

        if np.linalg.det(R) < 0:
            Vt[-1, :] *= -1
            R = Vt.T @ U.T

        return R

    @staticmethod
    def _apply_similarity_transform(points, scale, R, src_center, dst_center):
        pts = np.asarray(points, dtype=float)
        pts2 = pts * scale
        pts2 = pts2 - src_center
        pts2 = pts2 @ R.T
        pts2 = pts2 + dst_center
        return pts2

    @staticmethod
    def _compute_centerline_tangents(track_xy, closed=True):
        pts = np.asarray(track_xy, dtype=float)
        n = len(pts)
        tangents = np.zeros_like(pts)

        for i in range(n):
            prev_i = (i - 1) % n if closed else max(i - 1, 0)
            next_i = (i + 1) % n if closed else min(i + 1, n - 1)

            t = pts[next_i] - pts[prev_i]
            norm = np.linalg.norm(t)

            if norm < 1e-12:
                tangents[i] = np.array([1.0, 0.0])
            else:
                tangents[i] = t / norm

        return tangents

    @staticmethod
    def _compute_centerline_normals(track_xy, closed=True):
        tangents = FrenetConverter._compute_centerline_tangents(track_xy, closed=closed)
        normals = np.column_stack([-tangents[:, 1], tangents[:, 0]])
        return normals

    @staticmethod
    def _compute_centerline_curvature(track_xy, closed=True):
        pts = np.asarray(track_xy, dtype=float)
        n = len(pts)
        curvature = np.zeros(n, dtype=float)

        for i in range(n):
            prev_i = (i - 1) % n if closed else max(i - 1, 0)
            next_i = (i + 1) % n if closed else min(i + 1, n - 1)

            p_prev = pts[prev_i]
            p = pts[i]
            p_next = pts[next_i]

            a = np.linalg.norm(p - p_prev)
            b = np.linalg.norm(p_next - p)
            c = np.linalg.norm(p_next - p_prev)

            if a < 1e-12 or b < 1e-12 or c < 1e-12:
                curvature[i] = 0.0
                continue

            area2 = np.cross(p - p_prev, p_next - p_prev)
            curvature[i] = 2.0 * area2 / (a * b * c)

        return curvature

    @staticmethod
    def _timedelta_to_seconds(series):
        if pd.api.types.is_timedelta64_dtype(series):
            return series.dt.total_seconds()
        return pd.to_numeric(series, errors="coerce")

    @staticmethod
    def _safe_diff_over_time(values, time_sec):
        v = np.asarray(values, dtype=float)
        t = np.asarray(time_sec, dtype=float)

        out = np.full_like(v, np.nan, dtype=float)
        if len(v) == 0:
            return out
        if len(v) == 1:
            out[0] = 0.0
            return out

        for i in range(len(v)):
            if i == 0:
                dt = t[i + 1] - t[i]
                dv = v[i + 1] - v[i]
            elif i == len(v) - 1:
                dt = t[i] - t[i - 1]
                dv = v[i] - v[i - 1]
            else:
                dt = t[i + 1] - t[i - 1]
                dv = v[i + 1] - v[i - 1]

            if np.isfinite(dt) and abs(dt) > 1e-9:
                out[i] = dv / dt
            else:
                out[i] = 0.0

        return out

    @staticmethod
    def _unwrap_progress_to_s(progress_values, track_len):
        p = np.asarray(progress_values, dtype=float)
        if len(p) == 0:
            return np.array([], dtype=float)

        p_unwrapped = np.zeros_like(p)
        lap_offset = 0.0
        p_unwrapped[0] = p[0]

        for i in range(1, len(p)):
            cur = p[i]
            prev = p[i - 1]

            if prev > 0.8 and cur < 0.2:
                lap_offset += 1.0

            p_unwrapped[i] = cur + lap_offset

        return p_unwrapped * track_len

    def _build_segment_cache(self):
        pts = self.track_xy
        n = len(pts)

        A = pts
        B = np.roll(pts, -1, axis=0)
        AB = B - A

        seg_len = np.linalg.norm(AB, axis=1)
        seg_len_safe = np.where(seg_len < 1e-12, 1.0, seg_len)

        tangent = AB / seg_len_safe[:, None]
        normal = np.column_stack([-tangent[:, 1], tangent[:, 0]])

        self.seg_A = A
        self.seg_B = B
        self.seg_AB = AB
        self.seg_len = seg_len
        self.seg_len_safe = seg_len_safe
        self.seg_tangent = tangent
        self.seg_normal = normal
        self.seg_s_start = self.track_s_ref.copy()
        self.seg_count = n

    def _extract_xy_from_telemetry(self, telemetry):
        return telemetry[list(self.xy_cols)].dropna().to_numpy(dtype=float)

    def _extract_telemetry_df(self, telemetry):
        keep_cols = [
            "SessionTime", "Time", "Date", "Source",
            "X", "Y", "Z", "Status",
            "Speed", "RPM", "nGear", "Throttle", "Brake", "DRS",
            "Distance", "RelativeDistance", "DifferentialDistance",
            "DriverAhead", "DistanceToDriverAhead",
            "TrackStatus"
        ]
        cols = [c for c in keep_cols if c in telemetry.columns]

        df = telemetry[cols].copy()

        xy_cols = list(self.xy_cols)
        if not set(xy_cols).issubset(df.columns):
            raise ValueError(f"Telemetry missing required columns: {xy_cols}")

        df = df.dropna(subset=xy_cols).reset_index(drop=True)
        return df

    def extract_training_laps_from_session(
        self,
        session,
        drivers=None,
        lap_mode="fastest",
        min_points=50,
        max_laps=None
    ):
        laps_xy_list = []
        meta = []

        lap_df = session.laps.copy()

        if drivers is None:
            drivers = lap_df["Driver"].dropna().unique()

        count = 0
        for driver in drivers:
            try:
                driver_laps = lap_df.pick_drivers(driver)

                if lap_mode == "fastest":
                    lap = driver_laps.pick_fastest()
                    selected_laps = [lap] if lap is not None else []
                else:
                    raise ValueError("Only lap_mode='fastest' is implemented in this version.")

                for lap in selected_laps:
                    telemetry = lap.get_telemetry()
                    lap_xy = self._extract_xy_from_telemetry(telemetry)

                    if len(lap_xy) < min_points:
                        continue

                    laps_xy_list.append(lap_xy)
                    meta.append({
                        "driver": lap["Driver"],
                        "lap_number": lap.get("LapNumber", None)
                    })

                    count += 1
                    if max_laps is not None and count >= max_laps:
                        return laps_xy_list, meta

            except Exception as e:
                print(f"skip {driver}: {e}")

        return laps_xy_list, meta

    def fit_from_laps(self, laps_xy_list):
        align_rs, L_align = self._resample_closed_curve(self.align_xy, self.n_samples)
        align_center = align_rs.mean(axis=0)
        align0 = align_rs - align_center

        lap_rs_list = []
        scale_list = []

        for lap_xy in laps_xy_list:
            lap_rs, L_lap = self._resample_closed_curve(lap_xy, self.n_samples)
            scale = L_align / L_lap
            lap_rs_list.append(lap_rs)
            scale_list.append(scale)

        global_scale = float(np.median(scale_list))

        best_err = np.inf
        best = None

        for reverse in [False, True]:
            for shift in range(self.n_samples):
                A_all = []
                B_all = []
                src_centers = []

                for lap_rs in lap_rs_list:
                    lap_try = lap_rs[::-1] if reverse else lap_rs.copy()
                    lap_try = lap_try * global_scale
                    lap_try = np.roll(lap_try, shift=shift, axis=0)

                    lap_center = lap_try.mean(axis=0)
                    lap0 = lap_try - lap_center

                    A_all.append(lap0)
                    B_all.append(align0)
                    src_centers.append(lap_center)

                A_stack = np.vstack(A_all)
                B_stack = np.vstack(B_all)

                R = self._best_rotation_kabsch(A_stack, B_stack)
                A_rot = A_stack @ R.T
                err = np.mean(np.sum((A_rot - B_stack) ** 2, axis=1))

                if err < best_err:
                    best_err = err
                    best = {
                        "scale": global_scale,
                        "R": R,
                        "src_center": np.mean(src_centers, axis=0),
                        "dst_center": align_center,
                        "fit_error": err
                    }

        self.scale = best["scale"]
        self.R = best["R"]
        self.src_center = best["src_center"]
        self.dst_center = best["dst_center"]
        self.fit_error = best["fit_error"]
        self.is_fitted = True

    def fit(self, session, drivers=None, lap_mode="fastest", min_points=50, max_laps=None):
        laps_xy_list, meta = self.extract_training_laps_from_session(
            session=session,
            drivers=drivers,
            lap_mode=lap_mode,
            min_points=min_points,
            max_laps=max_laps
        )

        if len(laps_xy_list) == 0:
            raise RuntimeError("No usable laps extracted from session.")

        self.fit_from_laps(laps_xy_list)
        return meta

    def transform_xy(self, xy):
        if not self.is_fitted:
            raise RuntimeError("Call fit() first.")

        return self._apply_similarity_transform(
            xy,
            scale=self.scale,
            R=self.R,
            src_center=self.src_center,
            dst_center=self.dst_center
        )

    def transform_telemetry_df(self, telemetry_df):
        xy = telemetry_df[list(self.xy_cols)].to_numpy(dtype=float)
        aligned_xy = self.transform_xy(xy)

        out = telemetry_df.copy()
        out["x_aligned"] = aligned_xy[:, 0]
        out["y_aligned"] = aligned_xy[:, 1]
        return out

    def _candidate_window_indices(self, center_idx, radius):
        idxs = np.arange(center_idx - radius, center_idx + radius + 1)
        idxs = np.mod(idxs, self.seg_count)
        return idxs

    def _project_point_to_segments_vectorized(self, P, candidate_indices):
        A = self.seg_A[candidate_indices]
        AB = self.seg_AB[candidate_indices]
        seg_len = self.seg_len[candidate_indices]
        seg_normal = self.seg_normal[candidate_indices]
        seg_s_start = self.seg_s_start[candidate_indices]

        AP = P[None, :] - A
        denom = np.sum(AB * AB, axis=1)
        denom = np.where(denom < 1e-12, 1.0, denom)

        t = np.sum(AP * AB, axis=1) / denom
        t = np.clip(t, 0.0, 1.0)

        Q = A + t[:, None] * AB
        diff = P[None, :] - Q
        dist2 = np.sum(diff * diff, axis=1)

        best_local = np.argmin(dist2)
        seg_idx = candidate_indices[best_local]

        Q_best = Q[best_local]
        s_best = seg_s_start[best_local] + t[best_local] * seg_len[best_local]
        d_best = np.dot(P - Q_best, seg_normal[best_local])

        return Q_best, s_best, d_best, seg_idx

    def xy_to_sd_fast(self, aligned_xy, local_window_radius=8, global_every_n=200):
        aligned_xy = np.asarray(aligned_xy, dtype=float)
        rows = []

        prev_seg_idx = None

        for idx, P in enumerate(aligned_xy):
            if prev_seg_idx is None or (global_every_n is not None and idx % global_every_n == 0):
                candidate_indices = np.arange(self.seg_count)
            else:
                candidate_indices = self._candidate_window_indices(prev_seg_idx, local_window_radius)

            Q, s, d, seg_idx = self._project_point_to_segments_vectorized(P, candidate_indices)
            prev_seg_idx = seg_idx

            rows.append({
                "idx": idx,
                "x_aligned": P[0],
                "y_aligned": P[1],
                "proj_x": Q[0],
                "proj_y": Q[1],
                "s_m": s,
                "progress": (s % self.track_len) / self.track_len,
                "d_m": d,
                "seg_idx": seg_idx
            })

        return pd.DataFrame(rows)

    def telemetry_to_sd(self, telemetry_df, local_window_radius=8, global_every_n=200):
        if not {"x_aligned", "y_aligned"}.issubset(telemetry_df.columns):
            telemetry_df = self.transform_telemetry_df(telemetry_df)

        aligned_xy = telemetry_df[["x_aligned", "y_aligned"]].to_numpy(dtype=float)
        sd_df = self.xy_to_sd_fast(
            aligned_xy,
            local_window_radius=local_window_radius,
            global_every_n=global_every_n
        )

        out = pd.concat(
            [telemetry_df.reset_index(drop=True), sd_df.reset_index(drop=True)],
            axis=1
        )
        return out

    @staticmethod
    def smooth_d_series(d_values, window=9):
        s = pd.Series(d_values, dtype=float)
        smooth = s.rolling(window=window, center=True).mean()
        smooth = smooth.bfill().ffill()
        return smooth.to_numpy()

    def clip_d_to_track(self, d_values, seg_idx):
        left = self.track_df["w_tr_left_m"].to_numpy()
        right = self.track_df["w_tr_right_m"].to_numpy()

        seg_idx = np.asarray(seg_idx, dtype=int)
        w_left = left[seg_idx]
        w_right = right[seg_idx]

        d_clipped = np.minimum(np.maximum(d_values, -w_right), w_left)
        return d_clipped, w_left, w_right

    def postprocess_sd_time_order(self, sd_df, smooth_window=9):
        df = sd_df.copy()

        if "SessionTime" in df.columns:
            df = df.sort_values("SessionTime").copy()

        df["d_raw"] = df["d_m"]
        df["d_smooth"] = self.smooth_d_series(df["d_raw"].to_numpy(), window=smooth_window)

        d_final, w_left, w_right = self.clip_d_to_track(
            df["d_smooth"].to_numpy(),
            df["seg_idx"].to_numpy()
        )

        df["w_left"] = w_left
        df["w_right"] = w_right
        df["track_width"] = w_left + w_right
        df["d_final"] = d_final
        df["left_margin"] = w_left - d_final
        df["right_margin"] = w_right + d_final

        return df

    def reconstruct_reference_xy(self, processed_sd_df):
        df = processed_sd_df.copy()

        ref_x = []
        ref_y = []
        ref_heading = []
        curvature = []
        turn_sign = []
        tangent_x = []
        tangent_y = []
        normal_x = []
        normal_y = []

        for _, row in df.iterrows():
            seg_idx = int(row["seg_idx"])
            base_x = row["proj_x"]
            base_y = row["proj_y"]
            d = row["d_final"]

            nvec = self.track_normals[seg_idx]
            tvec = self.track_tangents[seg_idx]
            curv = self.track_curvature[seg_idx]

            x = base_x + d * nvec[0]
            y = base_y + d * nvec[1]

            ref_x.append(x)
            ref_y.append(y)
            ref_heading.append(np.arctan2(tvec[1], tvec[0]))
            curvature.append(curv)
            turn_sign.append(np.sign(curv))
            tangent_x.append(tvec[0])
            tangent_y.append(tvec[1])
            normal_x.append(nvec[0])
            normal_y.append(nvec[1])

        df["x_ref"] = ref_x
        df["y_ref"] = ref_y
        df["ref_heading"] = ref_heading
        df["curvature"] = curvature
        df["turn_sign"] = turn_sign
        df["tangent_x"] = tangent_x
        df["tangent_y"] = tangent_y
        df["normal_x"] = normal_x
        df["normal_y"] = normal_y

        return df

    def _add_time_dynamics(self, df):
        out = df.copy()

        if "SessionTime" not in out.columns:
            out["ds_dt"] = np.nan
            out["dd_dt"] = np.nan
            out["d2d_dt2"] = np.nan
            return out

        t_sec = self._timedelta_to_seconds(out["SessionTime"])
        s_unwrapped = self._unwrap_progress_to_s(out["progress"].to_numpy(), self.track_len)

        out["s_unwrapped_m"] = s_unwrapped
        out["ds_dt"] = self._safe_diff_over_time(s_unwrapped, t_sec)
        out["dd_dt"] = self._safe_diff_over_time(out["d_final"].to_numpy(dtype=float), t_sec)
        out["d2d_dt2"] = self._safe_diff_over_time(out["dd_dt"].to_numpy(dtype=float), t_sec)

        return out

    def process_full_stream(
        self,
        telemetry,
        smooth_window=9,
        resample_rule=None,
        local_window_radius=8,
        global_every_n=200
    ):
        if resample_rule is not None:
            telemetry = telemetry.resample_channels(rule=resample_rule)
            telemetry = telemetry.fill_missing()

        telemetry_df = self._extract_telemetry_df(telemetry)
        telemetry_df = self.transform_telemetry_df(telemetry_df)

        sd_df = self.telemetry_to_sd(
            telemetry_df,
            local_window_radius=local_window_radius,
            global_every_n=global_every_n
        )

        proc_df = self.postprocess_sd_time_order(sd_df, smooth_window=smooth_window)
        ref_df = self.reconstruct_reference_xy(proc_df)
        ref_df = self._add_time_dynamics(ref_df)

        if "SessionTime" in ref_df.columns:
            ref_df = ref_df.sort_values("SessionTime").reset_index(drop=True)

        return ref_df

    def _attach_lap_number_from_laps(self, full_df, driver_laps):
        out = full_df.copy()
        out["LapNumber"] = np.nan

        if "SessionTime" not in out.columns:
            return out

        st = out["SessionTime"]

        for _, lap in driver_laps.iterrows():
            lap_num = lap.get("LapNumber", np.nan)

            lap_start = None
            lap_end = None

            if "LapStartTime" in lap.index and pd.notna(lap["LapStartTime"]):
                lap_start = lap["LapStartTime"]

            if "Time" in lap.index and pd.notna(lap["Time"]):
                lap_end = lap["Time"]

            if lap_start is None or pd.isna(lap_start) or pd.isna(lap_end):
                continue

            mask = (st >= lap_start) & (st <= lap_end)
            out.loc[mask, "LapNumber"] = lap_num

        out["LapNumber"] = out["LapNumber"].ffill().bfill()
        return out

    def export_race_reference_csv(
        self,
        session,
        driver,
        out_csv_path,
        smooth_window=9,
        min_points=20,
        drop_offtrack=False,
        add_distance_channels=False,
        add_driver_ahead=False,
        resample_rule="200ms",
        local_window_radius=8,
        global_every_n=200
    ):
        driver_laps = session.laps.pick_drivers(driver)
        telemetry = driver_laps.get_telemetry()

        if len(telemetry) < min_points:
            raise RuntimeError(f"Telemetry too short for driver {driver}.")

        if add_distance_channels:
            try:
                if "DifferentialDistance" not in telemetry.columns:
                    telemetry = telemetry.add_differential_distance()
                if "Distance" not in telemetry.columns:
                    telemetry = telemetry.add_distance()
                if "RelativeDistance" not in telemetry.columns:
                    telemetry = telemetry.add_relative_distance()
            except Exception as e:
                print(f"warning: add distance channels failed for {driver}: {e}")

        if add_driver_ahead:
            try:
                if "DriverAhead" not in telemetry.columns or "DistanceToDriverAhead" not in telemetry.columns:
                    telemetry = telemetry.add_driver_ahead()
            except Exception as e:
                print(f"warning: add driver ahead failed for {driver}: {e}")

        race_df = self.process_full_stream(
            telemetry=telemetry,
            smooth_window=smooth_window,
            resample_rule=resample_rule,
            local_window_radius=local_window_radius,
            global_every_n=global_every_n
        )

        race_df["Driver"] = driver
        race_df = self._attach_lap_number_from_laps(race_df, driver_laps)

        if drop_offtrack and "Status" in race_df.columns:
            race_df = race_df[race_df["Status"] == "OnTrack"].copy()

        if "SessionTime" in race_df.columns:
            race_df = race_df.sort_values("SessionTime").reset_index(drop=True)

        race_df["sample_id"] = np.arange(len(race_df))

        preferred_cols = [
            "sample_id",
            "SessionTime", "Time", "Date",
            "Driver", "LapNumber",
            "X", "Y", "Z",
            "x_aligned", "y_aligned",
            "proj_x", "proj_y",
            "s_m", "s_unwrapped_m", "progress",
            "d_m", "d_raw", "d_smooth", "d_final",
            "w_left", "w_right", "track_width",
            "left_margin", "right_margin",
            "x_ref", "y_ref",
            "ref_heading",
            "curvature", "turn_sign",
            "tangent_x", "tangent_y",
            "normal_x", "normal_y",
            "ds_dt", "dd_dt", "d2d_dt2",
            "Speed", "RPM", "nGear", "Throttle", "Brake", "DRS",
            "Distance", "RelativeDistance", "DifferentialDistance",
            "DriverAhead", "DistanceToDriverAhead",
            "Status", "TrackStatus", "Source"
        ]

        out_cols = [c for c in preferred_cols if c in race_df.columns]
        out_df = race_df[out_cols].copy()
        out_df.to_csv(out_csv_path, index=False)
        return out_df

    def export_multi_driver_race_reference_csv(
        self,
        session,
        drivers,
        out_csv_path,
        smooth_window=9,
        min_points=20,
        drop_offtrack=False,
        add_distance_channels=False,
        add_driver_ahead=False,
        resample_rule="200ms",
        local_window_radius=8,
        global_every_n=200
    ):
        parts = []

        for driver in drivers:
            try:
                df_driver = self.export_race_reference_csv(
                    session=session,
                    driver=driver,
                    out_csv_path="/tmp/_ignore_single_driver.csv",
                    smooth_window=smooth_window,
                    min_points=min_points,
                    drop_offtrack=drop_offtrack,
                    add_distance_channels=add_distance_channels,
                    add_driver_ahead=add_driver_ahead,
                    resample_rule=resample_rule,
                    local_window_radius=local_window_radius,
                    global_every_n=global_every_n
                )
                parts.append(df_driver)
            except Exception as e:
                print(f"skip driver {driver}: {e}")

        if not parts:
            raise RuntimeError("No usable driver data found.")

        out_df = pd.concat(parts, ignore_index=True)

        if "SessionTime" in out_df.columns:
            out_df = out_df.sort_values(["SessionTime", "Driver"]).reset_index(drop=True)

        out_df["global_sample_id"] = np.arange(len(out_df))

        cols = ["global_sample_id"] + [c for c in out_df.columns if c != "global_sample_id"]
        out_df = out_df[cols].copy()
        out_df.to_csv(out_csv_path, index=False)
        return out_df

    def get_transform_params(self):
        if not self.is_fitted:
            raise RuntimeError("Call fit() first.")

        return {
            "scale": float(self.scale),
            "R": self.R.tolist(),
            "src_center": self.src_center.tolist(),
            "dst_center": self.dst_center.tolist(),
            "fit_error": float(self.fit_error)
        }

    def save_transform(self, path):
        params = self.get_transform_params()
        with open(path, "w", encoding="utf-8") as f:
            json.dump(params, f, indent=2, ensure_ascii=False)

    def load_transform(self, path):
        with open(path, "r", encoding="utf-8") as f:
            params = json.load(f)

        self.scale = float(params["scale"])
        self.R = np.array(params["R"], dtype=float)
        self.src_center = np.array(params["src_center"], dtype=float)
        self.dst_center = np.array(params["dst_center"], dtype=float)
        self.fit_error = float(params["fit_error"])
        self.is_fitted = True

In [ ]:
import os
import json
from urllib.request import urlretrieve

import pandas as pd
import fastf1

class RaceDataExtractor:
    """
    上层流程类:
        1) 读取 FastF1 session
        2) 下载赛道 centerline / raceline
        3) 创建输出目录
        4) 调用 FrenetConverter
        5) 批量导出一个或多个比赛的数据到 18738data
    """

    TRACK_DB_BASE = "https://raw.githubusercontent.com/TUMFTM/racetrack-database/master"

    def __init__(
        self,
        base_out_dir="18738data",
        cache_dir="fastf1_cache",
        xy_cols=("X", "Y"),
        n_samples=800,
        fit_lap_mode="fastest",
        fit_min_points=50,
        fit_max_laps=None,
        smooth_window=9,
        export_min_points=20,
        drop_offtrack=False,
        add_distance_channels=True,
        add_driver_ahead=True,
        resample_rule="200ms",
        local_window_radius=8,
        global_every_n=200
    ):
        self.base_out_dir = base_out_dir
        self.cache_dir = cache_dir
        self.xy_cols = xy_cols
        self.n_samples = n_samples

        self.fit_lap_mode = fit_lap_mode
        self.fit_min_points = fit_min_points
        self.fit_max_laps = fit_max_laps

        self.smooth_window = smooth_window
        self.export_min_points = export_min_points
        self.drop_offtrack = drop_offtrack
        self.add_distance_channels = add_distance_channels
        self.add_driver_ahead = add_driver_ahead
        self.resample_rule = resample_rule
        self.local_window_radius = local_window_radius
        self.global_every_n = global_every_n

        os.makedirs(self.base_out_dir, exist_ok=True)
        os.makedirs(self.cache_dir, exist_ok=True)

        fastf1.Cache.enable_cache(self.cache_dir)

    def load_session(self, year, grand_prix, session_type="R"):
        session = fastf1.get_session(year, grand_prix, session_type)
        session.load()
        return session

    def _download_file(self, url, local_path):
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        if not os.path.exists(local_path):
            urlretrieve(url, local_path)
        return local_path

    def load_track_assets(self, track_name, assets_dir):
        os.makedirs(assets_dir, exist_ok=True)

        track_url = f"{self.TRACK_DB_BASE}/tracks/{track_name}.csv"
        track_csv = os.path.join(assets_dir, f"{track_name}.csv")
        self._download_file(track_url, track_csv)

        raceline_url = f"{self.TRACK_DB_BASE}/racelines/{track_name}.csv"
        raceline_csv = os.path.join(assets_dir, f"{track_name}_raceline.csv")
        self._download_file(raceline_url, raceline_csv)

        track_df = pd.read_csv(track_csv)
        track_df.columns = track_df.columns.str.strip()
        track_df = track_df.rename(columns={"# x_m": "x_m"})

        raceline_df = pd.read_csv(raceline_csv)
        raceline_df.columns = (
            raceline_df.columns
            .str.strip()
            .str.replace("#", "", regex=False)
            .str.strip()
        )

        required_track_cols = ["x_m", "y_m", "w_tr_right_m", "w_tr_left_m"]
        for col in required_track_cols:
            if col not in track_df.columns:
                raise ValueError(f"Track CSV missing column: {col}")

        required_raceline_cols = ["x_m", "y_m"]
        for col in required_raceline_cols:
            if col not in raceline_df.columns:
                raise ValueError(f"Raceline CSV missing column: {col}")

        return {
            "track_df": track_df,
            "raceline_df": raceline_df,
            "track_csv": track_csv,
            "raceline_csv": raceline_csv
        }

    def build_converter(self, track_df, raceline_df):
        return FrenetConverter(
            track_df=track_df,
            raceline_df=raceline_df,
            xy_cols=self.xy_cols,
            n_samples=self.n_samples
        )

    def _make_output_dirs(self, track_name, year, session_type):
        session_key = f"{year}_{session_type}"
        session_dir = os.path.join(self.base_out_dir, track_name, session_key)
        drivers_dir = os.path.join(session_dir, "drivers")
        assets_dir = os.path.join(session_dir, "track_assets")

        os.makedirs(session_dir, exist_ok=True)
        os.makedirs(drivers_dir, exist_ok=True)
        os.makedirs(assets_dir, exist_ok=True)

        return {
            "session_dir": session_dir,
            "drivers_dir": drivers_dir,
            "assets_dir": assets_dir,
            "session_key": session_key
        }

    def process_one_job(self, job, drivers=None):
        """
        job 示例:
        {
            "track_name": "Silverstone",
            "year": 2025,
            "grand_prix": "Silverstone",
            "session_type": "R"
        }
        """
        track_name = job["track_name"]
        year = job["year"]
        grand_prix = job["grand_prix"]
        session_type = job.get("session_type", "R")

        out_dirs = self._make_output_dirs(track_name, year, session_type)

        print(f"\n=== Processing {track_name} | {year} | {session_type} ===")

        session = self.load_session(year, grand_prix, session_type)
        assets = self.load_track_assets(track_name, out_dirs["assets_dir"])

        converter = self.build_converter(
            track_df=assets["track_df"],
            raceline_df=assets["raceline_df"]
        )

        fit_meta = converter.fit(
            session=session,
            drivers=drivers,
            lap_mode=self.fit_lap_mode,
            min_points=self.fit_min_points,
            max_laps=self.fit_max_laps
        )

        transform_path = os.path.join(out_dirs["session_dir"], "transform_params.json")
        converter.save_transform(transform_path)

        if drivers is None:
            driver_list = session.laps["Driver"].dropna().unique().tolist()
        else:
            driver_list = list(drivers)

        saved_files = []
        failed_drivers = []

        for driver in driver_list:
            out_csv = os.path.join(out_dirs["drivers_dir"], f"{driver}.csv")
            try:
                converter.export_race_reference_csv(
                    session=session,
                    driver=driver,
                    out_csv_path=out_csv,
                    smooth_window=self.smooth_window,
                    min_points=self.export_min_points,
                    drop_offtrack=self.drop_offtrack,
                    add_distance_channels=self.add_distance_channels,
                    add_driver_ahead=self.add_driver_ahead,
                    resample_rule=self.resample_rule,
                    local_window_radius=self.local_window_radius,
                    global_every_n=self.global_every_n
                )
                saved_files.append(out_csv)
                print(f"saved {driver}: {out_csv}")
            except Exception as e:
                failed_drivers.append({"driver": driver, "error": str(e)})
                print(f"skip {driver}: {e}")

        meta = {
            "track_name": track_name,
            "year": year,
            "grand_prix": grand_prix,
            "session_type": session_type,
            "session_dir": out_dirs["session_dir"],
            "drivers_dir": out_dirs["drivers_dir"],
            "track_csv": assets["track_csv"],
            "raceline_csv": assets["raceline_csv"],
            "transform_path": transform_path,
            "n_saved_files": len(saved_files),
            "saved_files": saved_files,
            "failed_drivers": failed_drivers,
            "fit_meta": fit_meta
        }

        meta_path = os.path.join(out_dirs["session_dir"], "session_meta.json")
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)

        return meta

    def process_job_list(self, job_list, drivers=None):
        all_meta = []

        for job in job_list:
            try:
                meta = self.process_one_job(job=job, drivers=drivers)
                all_meta.append(meta)
            except Exception as e:
                fail_meta = {
                    "job": job,
                    "error": str(e)
                }
                all_meta.append(fail_meta)
                print(f"FAILED job {job}: {e}")

        summary_path = os.path.join(self.base_out_dir, "batch_summary.json")
        with open(summary_path, "w", encoding="utf-8") as f:
            json.dump(all_meta, f, ensure_ascii=False, indent=2)

        return all_meta

In [ ]:
import os
import glob
import itertools
import numpy as np
import pandas as pd


class RuleBasedPolicyDatasetBuilder:
    """
    从原始 driver csv 出发，自动生成：
        1) synced_states.csv
        2) pair_interactions.csv
        3) ego_interactions.csv
        4) full_policy_table.csv              # 调试用，含 interaction_label
        5) policy_training_table_deployable.csv  # 训练/部署用，不依赖中间标签做输入

    输入目录:
        folder/
            VER.csv
            HAM.csv
            NOR.csv
            ...

    也可以直接传某个 session 的 drivers 目录:
        18738data/Silverstone/2025_R/drivers/
    """

    def __init__(self, freq="200ms", tolerance="150ms"):
        self.freq = freq
        self.tolerance = tolerance

    # =========================================================
    # 1. 读取单车 CSV
    # =========================================================

    def load_driver_csv(self, csv_path: str) -> pd.DataFrame:
        df = pd.read_csv(csv_path)

        driver = os.path.splitext(os.path.basename(csv_path))[0].upper()
        df["Driver"] = driver

        if "SessionTime" in df.columns:
            df["SessionTime"] = pd.to_timedelta(df["SessionTime"], errors="coerce")

        if "Date" in df.columns:
            df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

        if "SessionTime" in df.columns:
            df = df.sort_values("SessionTime").reset_index(drop=True)

        # 补连续 s
        if "s_unwrapped_m" not in df.columns:
            if "LapNumber" in df.columns and "s_m" in df.columns:
                # 如果没有真正的连续 s，这里先退化用 s_m
                # 你后面如果想更严格，可以改成按 LapNumber 拼接
                df["s_unwrapped_m"] = pd.to_numeric(df["s_m"], errors="coerce")
            elif "progress" in df.columns and "track_width" in df.columns:
                # 没有 track_len 时只能先退化保留 progress
                df["s_unwrapped_m"] = pd.to_numeric(df["progress"], errors="coerce")
            elif "progress" in df.columns:
                df["s_unwrapped_m"] = pd.to_numeric(df["progress"], errors="coerce")
            else:
                raise ValueError(f"{csv_path} missing s_unwrapped_m / s_m / progress")

        numeric_cols = [
            "s_unwrapped_m", "s_m", "progress", "d_final",
            "x_ref", "y_ref", "Speed", "Throttle", "Brake",
            "curvature", "turn_sign", "left_margin", "right_margin",
            "track_width", "ds_dt", "dd_dt", "d2d_dt2"
        ]
        for c in numeric_cols:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")

        return df

    def load_all_driver_csvs(self, folder: str) -> dict:
        csv_files = sorted(glob.glob(os.path.join(folder, "*.csv")))
        if not csv_files:
            raise RuntimeError(f"No CSV files found in {folder}")

        data = {}
        for path in csv_files:
            try:
                df = self.load_driver_csv(path)
                driver = df["Driver"].iloc[0]
                data[driver] = df
                print(f"loaded {driver}: {len(df)} rows")
            except Exception as e:
                print(f"skip {path}: {e}")

        if not data:
            raise RuntimeError("No usable driver CSVs loaded.")

        return data

    # =========================================================
    # 2. 时间对齐
    # =========================================================

    def build_common_time_grid(self, driver_data: dict, freq: str = None) -> pd.DataFrame:
        if freq is None:
            freq = self.freq

        starts = []
        ends = []

        for _, df in driver_data.items():
            s = df["SessionTime"].dropna()
            if len(s) == 0:
                continue
            starts.append(s.min())
            ends.append(s.max())

        if not starts or not ends:
            raise RuntimeError("No valid SessionTime found.")

        global_start = min(starts)
        global_end = max(ends)

        time_grid = pd.DataFrame({
            "SessionTime": pd.timedelta_range(start=global_start, end=global_end, freq=freq)
        })
        return time_grid

    def resample_driver_to_grid(
        self,
        driver_df: pd.DataFrame,
        time_grid: pd.DataFrame,
        tolerance: str = None
    ) -> pd.DataFrame:
        if tolerance is None:
            tolerance = self.tolerance

        df = driver_df.sort_values("SessionTime").reset_index(drop=True).copy()
        grid = time_grid.sort_values("SessionTime").reset_index(drop=True).copy()

        tol = pd.Timedelta(tolerance)

        merged = pd.merge_asof(
            grid,
            df,
            on="SessionTime",
            direction="nearest",
            tolerance=tol
        )

        merged["is_valid"] = merged["Driver"].notna()
        return merged

    def build_synced_state_table(
        self,
        folder: str,
        freq: str = None,
        tolerance: str = None
    ) -> pd.DataFrame:
        driver_data = self.load_all_driver_csvs(folder)
        time_grid = self.build_common_time_grid(driver_data, freq=freq)

        parts = []
        for driver, df in driver_data.items():
            synced = self.resample_driver_to_grid(df, time_grid, tolerance=tolerance)
            synced["Driver"] = driver
            parts.append(synced)

        state_df = pd.concat(parts, ignore_index=True)
        state_df = state_df.sort_values(["SessionTime", "Driver"]).reset_index(drop=True)
        return state_df

    # =========================================================
    # 3. 交互规则
    # =========================================================

    def classify_interaction(
        self,
        delta_s: float,
        delta_d: float,
        delta_v: float,
        dd_i: float,
        dd_j: float,
        brake_i: float,
        brake_j: float,
        track_width: float = np.nan
    ) -> str:
        """
        以 i 为 ego，j 为 other
        delta_s = s_j - s_i
            > 0: j 在前
            < 0: j 在后

        delta_d = d_j - d_i
        delta_v = v_j - v_i
            > 0: j 更快
            < 0: i 更快
        """
        abs_ds = abs(delta_s)
        abs_dd = abs(delta_d)
        abs_dv = abs(delta_v)

        # 先排除明显不交互
        if abs_ds > 15 or abs_dd > 6:
            return "none"

        # 并排
        if abs_ds < 2.5 and 1.0 < abs_dd < 5.5:
            return "side_by_side"

        # 跟车：j 在前，横向较近，速度差不大
        if 2.5 <= delta_s <= 12 and abs_dd < 2.0 and abs_dv < 15:
            return "follow"

        # 逼近：j 在前，但 i 更快
        if 2.5 <= delta_s <= 12 and abs_dd < 2.5 and delta_v < -8:
            return "closing"

        # defend_candidate: j 在后方逼近，i 自己有明显横向移动
        if -12 <= delta_s <= -2.5 and abs_dd < 3.0 and abs(dd_i) > 1.0:
            return "defend_candidate"

        # avoid_candidate: 特别近，并且至少一方明显横向移动，或刹车
        if abs_ds < 4 and abs_dd < 2.5:
            if abs(dd_i) > 1.2 or abs(dd_j) > 1.2 or brake_i > 0 or brake_j > 0:
                return "avoid_candidate"

        return "none"

    # =========================================================
    # 4. pairwise 互动表
    # =========================================================

    def build_pairwise_interaction_table(self, state_df: pd.DataFrame) -> pd.DataFrame:
        rows = []

        for t, grp in state_df.groupby("SessionTime"):
            grp_valid = grp[grp["is_valid"] == True].copy()

            if len(grp_valid) < 2:
                continue

            recs = grp_valid.to_dict("records")

            for a, b in itertools.permutations(recs, 2):
                driver_i = a["Driver"]
                driver_j = b["Driver"]

                s_i = a.get("s_unwrapped_m", np.nan)
                s_j = b.get("s_unwrapped_m", np.nan)

                d_i = a.get("d_final", np.nan)
                d_j = b.get("d_final", np.nan)

                v_i = a.get("Speed", np.nan)
                v_j = b.get("Speed", np.nan)

                x_i = a.get("x_ref", np.nan)
                y_i = a.get("y_ref", np.nan)
                x_j = b.get("x_ref", np.nan)
                y_j = b.get("y_ref", np.nan)

                dd_i = a.get("dd_dt", np.nan)
                dd_j = b.get("dd_dt", np.nan)

                brake_i = a.get("Brake", 0)
                brake_j = b.get("Brake", 0)

                track_width = a.get("track_width", np.nan)

                if any(pd.isna(x) for x in [s_i, s_j, d_i, d_j, v_i, v_j, x_i, y_i, x_j, y_j]):
                    continue

                delta_s = s_j - s_i
                delta_d = d_j - d_i
                delta_v = v_j - v_i
                euclid_dist = np.sqrt((x_j - x_i) ** 2 + (y_j - y_i) ** 2)

                label = self.classify_interaction(
                    delta_s=delta_s,
                    delta_d=delta_d,
                    delta_v=delta_v,
                    dd_i=0 if pd.isna(dd_i) else dd_i,
                    dd_j=0 if pd.isna(dd_j) else dd_j,
                    brake_i=0 if pd.isna(brake_i) else brake_i,
                    brake_j=0 if pd.isna(brake_j) else brake_j,
                    track_width=track_width
                )

                rows.append({
                    "SessionTime": t,
                    "Driver_i": driver_i,
                    "Driver_j": driver_j,
                    "delta_s": delta_s,
                    "delta_d": delta_d,
                    "delta_v": delta_v,
                    "euclid_dist": euclid_dist,
                    "label": label
                })

        pair_df = pd.DataFrame(rows)
        if len(pair_df) == 0:
            return pair_df

        pair_df = pair_df.sort_values(["SessionTime", "Driver_i", "Driver_j"]).reset_index(drop=True)
        return pair_df

    # =========================================================
    # 5. ego 摘要
    # =========================================================

    def build_ego_interaction_summary(self, pair_df: pd.DataFrame) -> pd.DataFrame:
        """
        对每个时刻、每个 Driver_i，挑最重要的一条交互
        优先级:
            side_by_side > closing > defend_candidate > avoid_candidate > follow > none
        """
        if len(pair_df) == 0:
            return pair_df

        priority = {
            "side_by_side": 5,
            "closing": 4,
            "defend_candidate": 3,
            "avoid_candidate": 3,
            "follow": 2,
            "none": 0
        }

        df = pair_df.copy()
        df["priority"] = df["label"].map(priority).fillna(0)

        df = df.sort_values(
            ["SessionTime", "Driver_i", "priority", "euclid_dist"],
            ascending=[True, True, False, True]
        )

        ego_df = df.groupby(["SessionTime", "Driver_i"], as_index=False).first()
        ego_df = ego_df.rename(columns={
            "Driver_i": "Driver",
            "Driver_j": "TargetDriver",
            "label": "interaction_label"
        })

        keep_cols = [
            "SessionTime", "Driver", "TargetDriver",
            "interaction_label",
            "delta_s", "delta_d", "delta_v", "euclid_dist"
        ]
        ego_df = ego_df[keep_cols].copy()
        return ego_df

    # =========================================================
    # 6. interaction -> action
    # =========================================================

    def interaction_to_action(self, row) -> str:
        """
        规则老师：
        输入当前局面，输出动作标签

        动作空间:
            keep
            block_left
            block_right
            avoid_left
            avoid_right
        """
        interaction = row.get("interaction_label", row.get("label", "none"))
        delta_s = row.get("delta_s", np.nan)
        delta_d = row.get("delta_d", np.nan)
        delta_v = row.get("delta_v", np.nan)
        _ = row.get("euclid_dist", np.nan)

        # 自车状态
        _ = row.get("Speed", np.nan)
        _ = row.get("dd_dt", 0.0)
        _ = row.get("turn_sign", 0.0)
        left_margin = row.get("left_margin", np.nan)
        right_margin = row.get("right_margin", np.nan)

        # 安全边界
        if pd.isna(left_margin):
            left_margin = 999.0
        if pd.isna(right_margin):
            right_margin = 999.0

        # 没交互 => keep
        if interaction == "none" or pd.isna(delta_s) or pd.isna(delta_d):
            return "keep"

        # delta_d > 0 表示 other 在正侧
        other_on_positive_side = (delta_d > 0)

        # 1) follow: 正常跟车
        if interaction == "follow":
            return "keep"

        # 2) closing: 对手在前，自己在逼近，偏防守
        if interaction == "closing":
            if other_on_positive_side:
                if left_margin > 0.8:
                    return "block_left"
                return "keep"
            else:
                if right_margin > 0.8:
                    return "block_right"
                return "keep"

        # 3) side_by_side: 并排优先规避
        if interaction == "side_by_side":
            if other_on_positive_side:
                if right_margin > 0.8:
                    return "avoid_right"
                return "keep"
            else:
                if left_margin > 0.8:
                    return "avoid_left"
                return "keep"

        # 4) defend_candidate: 倾向 block
        if interaction == "defend_candidate":
            if other_on_positive_side:
                if left_margin > 0.8:
                    return "block_left"
                return "keep"
            else:
                if right_margin > 0.8:
                    return "block_right"
                return "keep"

        # 5) avoid_candidate: 倾向 avoid
        if interaction == "avoid_candidate":
            if other_on_positive_side:
                if right_margin > 0.8:
                    return "avoid_right"
                return "keep"
            else:
                if left_margin > 0.8:
                    return "avoid_left"
                return "keep"

        return "keep"

    def add_action_labels(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        out["action_label"] = out.apply(self.interaction_to_action, axis=1)
        return out

    # =========================================================
    # 7. merge 成训练表
    # =========================================================

    def build_training_base(
        self,
        state_df: pd.DataFrame,
        ego_df: pd.DataFrame
    ) -> pd.DataFrame:
        states = state_df.copy()
        ego = ego_df.copy()

        if "SessionTime" in states.columns:
            states["SessionTime"] = pd.to_timedelta(states["SessionTime"], errors="coerce")
        if "SessionTime" in ego.columns:
            ego["SessionTime"] = pd.to_timedelta(ego["SessionTime"], errors="coerce")

        df = ego.merge(
            states,
            on=["SessionTime", "Driver"],
            how="left",
            suffixes=("", "_state")
        )

        numeric_cols = [
            "delta_s", "delta_d", "delta_v", "euclid_dist",
            "Speed", "Throttle", "Brake",
            "d_final", "ds_dt", "dd_dt", "d2d_dt2",
            "curvature", "turn_sign",
            "left_margin", "right_margin",
            "track_width",
            "s_unwrapped_m", "s_m", "progress"
        ]
        for c in numeric_cols:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")

        return df

    def build_full_labeled_table(
        self,
        state_df: pd.DataFrame,
        ego_df: pd.DataFrame
    ) -> pd.DataFrame:
        """
        调试用全量表：
        包含 interaction_label 和 action_label
        """
        full_df = self.build_training_base(state_df, ego_df)
        full_df = self.add_action_labels(full_df)
        return full_df

    # =========================================================
    # 7.5 部署友好训练表
    # =========================================================

    def get_model_feature_columns(self):
        """
        这些列必须都是 Unity 运行时能直接算出来的。
        不依赖 interaction_label / TargetDriver / action_label。
        """
        return [
            "delta_s",
            "delta_d",
            "delta_v",
            "euclid_dist",
            "Speed",
            "Throttle",
            "Brake",
            "d_final",
            "ds_dt",
            "dd_dt",
            "d2d_dt2",
            "curvature",
            "turn_sign",
            "left_margin",
            "right_margin",
            "track_width"
        ]

    def build_deployable_policy_table(
        self,
        state_df: pd.DataFrame,
        ego_df: pd.DataFrame
    ) -> pd.DataFrame:
        """
        先用 interaction_label 生成 action_label，
        但最终导出的训练表只保留:
            - Unity可实时计算的输入特征
            - action_label
        """
        full_df = self.build_full_labeled_table(state_df, ego_df)

        feature_cols = [c for c in self.get_model_feature_columns() if c in full_df.columns]
        keep_cols = ["SessionTime", "Driver"] + feature_cols + ["action_label"]

        deploy_df = full_df[keep_cols].copy()
        deploy_df = deploy_df.dropna(subset=["action_label"]).reset_index(drop=True)

        return deploy_df

    # =========================================================
    # 8. 一键处理
    # =========================================================

    def build_from_folder(
        self,
        folder: str,
        freq: str = None,
        tolerance: str = None,
        out_state_csv: str = None,
        out_pair_csv: str = None,
        out_ego_csv: str = None,
        out_full_policy_csv: str = None,
        out_policy_csv: str = None
    ):
        print("Loading and syncing driver states...")
        state_df = self.build_synced_state_table(
            folder=folder,
            freq=freq,
            tolerance=tolerance
        )
        print("synced states:", state_df.shape)

        print("Building pairwise interaction table...")
        pair_df = self.build_pairwise_interaction_table(state_df)
        print("pair interactions:", pair_df.shape)

        print("Building ego interaction summary...")
        ego_df = self.build_ego_interaction_summary(pair_df)
        print("ego summary:", ego_df.shape)

        print("Building full labeled policy table...")
        full_policy_df = self.build_full_labeled_table(state_df, ego_df)
        print("full labeled policy table:", full_policy_df.shape)

        print("Building deployable policy training table...")
        policy_df = self.build_deployable_policy_table(state_df, ego_df)
        print("deployable policy training table:", policy_df.shape)

        if out_state_csv is not None:
            state_df.to_csv(out_state_csv, index=False)
            print("saved:", out_state_csv)

        if out_pair_csv is not None:
            pair_df.to_csv(out_pair_csv, index=False)
            print("saved:", out_pair_csv)

        if out_ego_csv is not None:
            ego_df.to_csv(out_ego_csv, index=False)
            print("saved:", out_ego_csv)

        if out_full_policy_csv is not None:
            full_policy_df.to_csv(out_full_policy_csv, index=False)
            print("saved:", out_full_policy_csv)

        if out_policy_csv is not None:
            policy_df.to_csv(out_policy_csv, index=False)
            print("saved:", out_policy_csv)

        return state_df, pair_df, ego_df, full_policy_df, policy_df

In [ ]:
import json
import os

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


class TabularPolicyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class MLPPolicyNet(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.net(x)


class MLPPolicyTrainer:
    def __init__(
        self,
        feature_cols=None,
        label_col="action_label",
        batch_size=256,
        lr=1e-3,
        epochs=20,
        random_state=42,
        device=None
    ):
        self.feature_cols = feature_cols
        self.label_col = label_col
        self.batch_size = batch_size
        self.lr = lr
        self.epochs = epochs
        self.random_state = random_state

        if device is None:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
        else:
            self.device = device

        self.imputer = None
        self.scaler = None
        self.label_encoder = None
        self.model = None
        self.meta = None

    def default_feature_cols(self):
        return [
            "delta_s",
            "delta_d",
            "delta_v",
            "euclid_dist",
            "Speed",
            "Throttle",
            "Brake",
            "d_final",
            "ds_dt",
            "dd_dt",
            "d2d_dt2",
            "curvature",
            "turn_sign",
            "left_margin",
            "right_margin",
            "track_width"
        ]

    def load_table(self, csv_path):
        df = pd.read_csv(csv_path)
        return df

    def prepare_data(self, df):
        if self.feature_cols is None:
            self.feature_cols = self.default_feature_cols()

        feature_cols = [c for c in self.feature_cols if c in df.columns]

        train_df = df.dropna(subset=[self.label_col]).copy()

        X = train_df[feature_cols].copy()
        y = train_df[self.label_col].copy()

        self.imputer = SimpleImputer(strategy="median")
        X_imp = self.imputer.fit_transform(X)

        self.scaler = StandardScaler()
        X_scaled = self.scaler.fit_transform(X_imp)

        self.label_encoder = LabelEncoder()
        y_enc = self.label_encoder.fit_transform(y)

        return X_scaled, y_enc, feature_cols, train_df

    def train(self, csv_path, model_out_path="policy_mlp.pt", meta_out_path="policy_mlp_meta.json"):
        df = self.load_table(csv_path)
        X, y, feature_cols, train_df = self.prepare_data(df)

        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.2,
            random_state=self.random_state,
            stratify=y
        )

        train_dataset = TabularPolicyDataset(X_train, y_train)
        test_dataset = TabularPolicyDataset(X_test, y_test)

        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=self.batch_size, shuffle=False)

        input_dim = X_train.shape[1]
        num_classes = len(np.unique(y))

        self.model = MLPPolicyNet(input_dim=input_dim, num_classes=num_classes).to(self.device)

        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)

        for epoch in range(self.epochs):
            self.model.train()
            total_loss = 0.0

            for xb, yb in train_loader:
                xb = xb.to(self.device)
                yb = yb.to(self.device)

                optimizer.zero_grad()
                logits = self.model(xb)
                loss = criterion(logits, yb)
                loss.backward()
                optimizer.step()

                total_loss += loss.item() * xb.size(0)

            avg_loss = total_loss / len(train_dataset)
            print(f"Epoch {epoch+1}/{self.epochs} | loss = {avg_loss:.6f}")

        y_pred = self.predict_numpy(X_test)
        acc = accuracy_score(y_test, y_pred)

        print("\nAccuracy:", acc)
        print("\nClassification Report:")
        print(classification_report(
            y_test,
            y_pred,
            target_names=self.label_encoder.classes_
        ))

        print("\nConfusion Matrix:")
        print(confusion_matrix(y_test, y_pred))

        torch.save(self.model.state_dict(), model_out_path)

        preprocess_bundle = {
            "feature_cols": feature_cols,
            "imputer": self.imputer,
            "scaler": self.scaler,
            "label_encoder": self.label_encoder
        }
        joblib.dump(preprocess_bundle, model_out_path.replace(".pt", "_preprocess.joblib"))

        self.meta = {
            "model_type": "MLPPolicyNet",
            "feature_cols": feature_cols,
            "classes": self.label_encoder.classes_.tolist(),
            "accuracy": float(acc),
            "n_samples": int(len(train_df)),
            "device": self.device
        }

        with open(meta_out_path, "w", encoding="utf-8") as f:
            json.dump(self.meta, f, ensure_ascii=False, indent=2)

        print(f"\nSaved model to: {model_out_path}")
        print(f"Saved preprocess to: {model_out_path.replace('.pt', '_preprocess.joblib')}")
        print(f"Saved meta to: {meta_out_path}")

        return {
            "model": self.model,
            "meta": self.meta,
            "feature_cols": feature_cols
        }

    def predict_numpy(self, X_np):
        self.model.eval()
        preds = []

        with torch.no_grad():
            X_tensor = torch.tensor(X_np, dtype=torch.float32).to(self.device)
            logits = self.model(X_tensor)
            pred = torch.argmax(logits, dim=1).cpu().numpy()
            preds.extend(pred.tolist())

        return np.array(preds)

    def predict_one(self, feature_row: dict):
        row = {}
        for c in self.feature_cols:
            row[c] = feature_row.get(c, np.nan)

        X = pd.DataFrame([row], columns=self.feature_cols)
        X_imp = self.imputer.transform(X)
        X_scaled = self.scaler.transform(X_imp)

        pred_enc = self.predict_numpy(X_scaled)[0]
        pred_label = self.label_encoder.inverse_transform([pred_enc])[0]
        return pred_label

In [ ]:
import os
import glob
import random
import pandas as pd

# =========================================================
# 1. 配置：准备大约 20 场比赛
# =========================================================

SESSION_JOBS = [
    {"track_name": "Silverstone", "year": 2025, "grand_prix": "Silverstone", "session_type": "R"},
    {"track_name": "Monza",       "year": 2025, "grand_prix": "Monza",       "session_type": "R"},
    {"track_name": "Spa",         "year": 2025, "grand_prix": "Spa",         "session_type": "R"},
    {"track_name": "Suzuka",      "year": 2025, "grand_prix": "Suzuka",      "session_type": "R"},
    {"track_name": "Austin",      "year": 2025, "grand_prix": "Austin",      "session_type": "R"},
    {"track_name": "Budapest",    "year": 2025, "grand_prix": "Hungary",     "session_type": "R"},
    {"track_name": "Catalunya",   "year": 2025, "grand_prix": "Spain",       "session_type": "R"},
    {"track_name": "Mexico City", "year": 2025, "grand_prix": "Mexico City", "session_type": "R"},
    {"track_name": "Montreal",    "year": 2025, "grand_prix": "Canada",      "session_type": "R"},
    {"track_name": "Sakhir",      "year": 2025, "grand_prix": "Bahrain",     "session_type": "R"},
    {"track_name": "Sao Paulo",   "year": 2025, "grand_prix": "Brazil",      "session_type": "R"},
    {"track_name": "Shanghai",    "year": 2025, "grand_prix": "China",       "session_type": "R"},
    {"track_name": "Spielberg",   "year": 2025, "grand_prix": "Austria",     "session_type": "R"},
    {"track_name": "Yas Marina",  "year": 2025, "grand_prix": "Abu Dhabi",   "session_type": "R"},
    {"track_name": "Melbourne",   "year": 2025, "grand_prix": "Australia",   "session_type": "R"},
    {"track_name": "Silverstone", "year": 2024, "grand_prix": "Silverstone", "session_type": "R"},
    {"track_name": "Monza",       "year": 2024, "grand_prix": "Monza",       "session_type": "R"},
    {"track_name": "Spa",         "year": 2024, "grand_prix": "Spa",         "session_type": "R"},
    {"track_name": "Suzuka",      "year": 2024, "grand_prix": "Suzuka",      "session_type": "R"},
    {"track_name": "Austin",      "year": 2024, "grand_prix": "Austin",      "session_type": "R"},
]

BASE_OUT_DIR = "/content/18738data"

# =========================================================
# 2. 批量提取原始 driver csv
# =========================================================

extractor = RaceDataExtractor(
    base_out_dir=BASE_OUT_DIR,
    cache_dir="/content/fastf1_cache",
    resample_rule="200ms",
    add_distance_channels=True,
    add_driver_ahead=True
)

extract_summary = extractor.process_job_list(SESSION_JOBS)

print("\nRaw extraction done.")
print(f"Processed jobs: {len(extract_summary)}")

# =========================================================
# 3. 对每场比赛做规则标注
# =========================================================

builder = RuleBasedPolicyDatasetBuilder(
    freq="200ms",
    tolerance="150ms"
)

all_policy_tables = []
all_policy_tables_interaction_only = []

for job in SESSION_JOBS:
    track_name = job["track_name"]
    year = job["year"]
    session_type = job["session_type"]

    session_dir = os.path.join(BASE_OUT_DIR, track_name, f"{year}_{session_type}")
    drivers_dir = os.path.join(session_dir, "drivers")

    if not os.path.exists(drivers_dir):
        print(f"skip missing drivers dir: {drivers_dir}")
        continue

    try:
        print(f"\n=== Building labels for {track_name} {year} {session_type} ===")

        state_df, pair_df, ego_df, full_policy_df, policy_df = builder.build_from_folder(
            folder=drivers_dir,
            out_state_csv=os.path.join(session_dir, "synced_states.csv"),
            out_pair_csv=os.path.join(session_dir, "pair_interactions.csv"),
            out_ego_csv=os.path.join(session_dir, "ego_interactions.csv"),
            out_full_policy_csv=os.path.join(session_dir, "full_policy_table.csv"),
            out_policy_csv=os.path.join(session_dir, "policy_training_table_deployable.csv")
        )

        # 给总表增加来源信息
        policy_df = policy_df.copy()
        policy_df["track_name"] = track_name
        policy_df["year"] = year
        policy_df["session_type"] = session_type

        all_policy_tables.append(policy_df)

        # 只保留有交互动作（非 keep）
        interaction_only_df = policy_df[policy_df["action_label"] != "keep"].copy()
        all_policy_tables_interaction_only.append(interaction_only_df)

    except Exception as e:
        print(f"FAILED labeling {track_name} {year} {session_type}: {e}")

# =========================================================
# 4. 合并所有比赛训练表
# =========================================================

train_dataset_dir = os.path.join(BASE_OUT_DIR, "train_dataset")
os.makedirs(train_dataset_dir, exist_ok=True)

if len(all_policy_tables) > 0:
    all_policy_df = pd.concat(all_policy_tables, ignore_index=True)
    all_policy_path = os.path.join(train_dataset_dir, "all_policy_training_table_deployable.csv")
    all_policy_df.to_csv(all_policy_path, index=False)
    print(f"\nSaved full merged deployable training table: {all_policy_path}")
    print(all_policy_df["action_label"].value_counts())

if len(all_policy_tables_interaction_only) > 0:
    all_interaction_only_df = pd.concat(all_policy_tables_interaction_only, ignore_index=True)
    interaction_only_path = os.path.join(train_dataset_dir, "all_policy_training_table_interaction_only.csv")
    all_interaction_only_df.to_csv(interaction_only_path, index=False)
    print(f"\nSaved interaction-only training table: {interaction_only_path}")
    print(all_interaction_only_df["action_label"].value_counts())

# =========================================================
# 5. 再生成一个“keep下采样版”
# =========================================================

def downsample_keep_rows(df, keep_ratio=0.2, random_state=42):
    keep_df = df[df["action_label"] == "keep"].copy()
    non_keep_df = df[df["action_label"] != "keep"].copy()

    if len(keep_df) > 0:
        keep_df = keep_df.sample(frac=keep_ratio, random_state=random_state)

    out_df = pd.concat([keep_df, non_keep_df], ignore_index=True)
    out_df = out_df.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    return out_df

if len(all_policy_tables) > 0:
    balanced_policy_df = downsample_keep_rows(all_policy_df, keep_ratio=0.2, random_state=42)
    balanced_path = os.path.join(train_dataset_dir, "all_policy_training_table_keep20.csv")
    balanced_policy_df.to_csv(balanced_path, index=False)

    print(f"\nSaved keep-downsampled training table: {balanced_path}")
    print(balanced_policy_df["action_label"].value_counts())


=== Processing Silverstone | 2025 | R ===


core           INFO 	Loading data for British Grand Prix - Race [v3.8.2]
INFO:fastf1.fastf1.core:Loading data for British Grand Prix - Race [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_status_d

saved VER: /content/18738data/Silverstone/2025_R/drivers/VER.csv
saved GAS: /content/18738data/Silverstone/2025_R/drivers/GAS.csv
saved ANT: /content/18738data/Silverstone/2025_R/drivers/ANT.csv
saved ALO: /content/18738data/Silverstone/2025_R/drivers/ALO.csv
saved LEC: /content/18738data/Silverstone/2025_R/drivers/LEC.csv
saved STR: /content/18738data/Silverstone/2025_R/drivers/STR.csv
saved TSU: /content/18738data/Silverstone/2025_R/drivers/TSU.csv
saved ALB: /content/18738data/Silverstone/2025_R/drivers/ALB.csv
saved HUL: /content/18738data/Silverstone/2025_R/drivers/HUL.csv
saved LAW: /content/18738data/Silverstone/2025_R/drivers/LAW.csv
saved OCO: /content/18738data/Silverstone/2025_R/drivers/OCO.csv
saved NOR: /content/18738data/Silverstone/2025_R/drivers/NOR.csv
saved COL: /content/18738data/Silverstone/2025_R/drivers/COL.csv
saved HAM: /content/18738data/Silverstone/2025_R/drivers/HAM.csv
saved BOR: /content/18738data/Silverstone/2025_R/drivers/BOR.csv
saved SAI: /content/18738

core           INFO 	Loading data for Italian Grand Prix - Race [v3.8.2]
INFO:fastf1.fastf1.core:Loading data for Italian Grand Prix - Race [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...


saved BEA: /content/18738data/Silverstone/2025_R/drivers/BEA.csv

=== Processing Monza | 2025 | R ===


...

req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
INFO:fastf1.api:Fetching session status data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
INFO:fastf1.fastf1.req:No cached data found for lap_count. Loading data...
_api           INFO 	F

saved VER: /content/18738data/Suzuka/2024_R/drivers/VER.csv
saved PER: /content/18738data/Suzuka/2024_R/drivers/PER.csv
saved SAI: /content/18738data/Suzuka/2024_R/drivers/SAI.csv
saved LEC: /content/18738data/Suzuka/2024_R/drivers/LEC.csv
saved NOR: /content/18738data/Suzuka/2024_R/drivers/NOR.csv
saved ALO: /content/18738data/Suzuka/2024_R/drivers/ALO.csv
saved RUS: /content/18738data/Suzuka/2024_R/drivers/RUS.csv
saved PIA: /content/18738data/Suzuka/2024_R/drivers/PIA.csv
saved HAM: /content/18738data/Suzuka/2024_R/drivers/HAM.csv
saved TSU: /content/18738data/Suzuka/2024_R/drivers/TSU.csv
saved HUL: /content/18738data/Suzuka/2024_R/drivers/HUL.csv
saved STR: /content/18738data/Suzuka/2024_R/drivers/STR.csv
saved MAG: /content/18738data/Suzuka/2024_R/drivers/MAG.csv
saved BOT: /content/18738data/Suzuka/2024_R/drivers/BOT.csv
saved OCO: /content/18738data/Suzuka/2024_R/drivers/OCO.csv
saved GAS: /content/18738data/Suzuka/2024_R/drivers/GAS.csv
saved SAR: /content/18738data/Suzuka/202

core           INFO 	Loading data for United States Grand Prix - Race [v3.8.2]
INFO:fastf1.fastf1.core:Loading data for United States Grand Prix - Race [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for sess

saved VER: /content/18738data/Austin/2024_R/drivers/VER.csv
saved GAS: /content/18738data/Austin/2024_R/drivers/GAS.csv
saved PER: /content/18738data/Austin/2024_R/drivers/PER.csv
saved ALO: /content/18738data/Austin/2024_R/drivers/ALO.csv
saved LEC: /content/18738data/Austin/2024_R/drivers/LEC.csv
saved STR: /content/18738data/Austin/2024_R/drivers/STR.csv
saved MAG: /content/18738data/Austin/2024_R/drivers/MAG.csv
saved TSU: /content/18738data/Austin/2024_R/drivers/TSU.csv
saved ALB: /content/18738data/Austin/2024_R/drivers/ALB.csv
saved ZHO: /content/18738data/Austin/2024_R/drivers/ZHO.csv
saved HUL: /content/18738data/Austin/2024_R/drivers/HUL.csv
saved LAW: /content/18738data/Austin/2024_R/drivers/LAW.csv
saved OCO: /content/18738data/Austin/2024_R/drivers/OCO.csv
saved NOR: /content/18738data/Austin/2024_R/drivers/NOR.csv
saved COL: /content/18738data/Austin/2024_R/drivers/COL.csv
saved HAM: /content/18738data/Austin/2024_R/drivers/HAM.csv
saved SAI: /content/18738data/Austin/202

KeyboardInterrupt: 

In [ ]:
!zip -r /content/18738data.zip /content/18738data

  adding: content/18738data/ (stored 0%)
  adding: content/18738data/Sao Paulo/ (stored 0%)
  adding: content/18738data/Sao Paulo/2025_R/ (stored 0%)
  adding: content/18738data/Sao Paulo/2025_R/track_assets/ (stored 0%)
  adding: content/18738data/Sao Paulo/2025_R/drivers/ (stored 0%)
  adding: content/18738data/Silverstone/ (stored 0%)
  adding: content/18738data/Silverstone/2025_R/ (stored 0%)
  adding: content/18738data/Silverstone/2025_R/session_meta.json (deflated 85%)
  adding: content/18738data/Silverstone/2025_R/policy_training_table_deployable.csv (deflated 63%)
  adding: content/18738data/Silverstone/2025_R/synced_states.csv (deflated 64%)
  adding: content/18738data/Silverstone/2025_R/full_policy_table.csv (deflated 64%)
  adding: content/18738data/Silverstone/2025_R/track_assets/ (stored 0%)
  adding: content/18738data/Silverstone/2025_R/track_assets/Silverstone.csv (deflated 52%)
  adding: content/18738data/Silverstone/2025_R/track_assets/Silverstone_raceline.csv (deflate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!cp /content/18738data.zip /content/drive/MyDrive/

In [ ]:
!find /content/drive/MyDrive -maxdepth 3 -name "18738data.zip"

/content/drive/MyDrive/18738data.zip
